In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
from src.config import *

from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None

In [2]:
REGION_LIST = ['강릉', '대관령']

In [3]:
def load_dataset(region) -> tuple:
    """
    [입력 가공]
    region: REGION_LIST 리스트에 저장된 지역
    df = 앞 단계에서 전처리된 지역별 데이터 셋 csv

    [1차가공]
    train_df = df에서 연도가 2018년 이하인 데이터
    test_df = df에서 연도가 2019년 이상인 데이터

    [2차가공]
    - date, price_change_rate, target 컬럼을 feature에서 제외
    - target 컬럼을 y_train, y_test로 분리
    - 결과적으로 X_train, X_test는 모델 입력용 feature 데이터
    - y_train, y_test는 정답 레이블 데이터

    [출력]
    X_train, X_test: 모델 입력용 feature DataFrame
    y_train, y_test: target 레이블 Series
    """
    df = pd.read_csv('processed/dataset_' + region + '.csv')
    df['date'] = pd.to_datetime(df['date'])

    train = df[df['date'].dt.year <= 2018]
    test = df[df['date'].dt.year >= 2019]

    print(f"\n{'='*40}")
    print(f"  [{region}] 데이터셋 로드")
    print(f"{'='*40}")
    print(f"학습: {len(train)}행, 테스트: {len(test)}행")

    nan_train = train.isnull().sum()
    nan_test = test.isnull().sum()

    print(f"\n[train NaN] {'없음' if not nan_train.any() else ''}")
    if nan_train.any():
        print(nan_train[nan_train > 0])

    print(f"\n[test NaN] {'없음' if not nan_test.any() else ''}")
    if nan_test.any():
        print(nan_test[nan_test > 0])

    # 전처리
    # 현재 데이터 프레임에서 date와 price_change_rate 컬럼을 제외시키기
    exclude_cols = ['date', 'price_change_rate']
    target_col = 'target'

    # 학습용 데이터와 테스트용 데이터를 분리
    X_train = train.drop(columns=exclude_cols + [target_col])
    X_test = test.drop(columns=exclude_cols + [target_col])
    y_train = train[target_col]
    y_test = test[target_col]

    return X_train, X_test, y_train, y_test

In [4]:
def scale_for_svm(X_train, X_test):
    # MinMaxScaler는 SVM 모델에만 사용한다.
    # SVM은 데이터 포인트 간 거리를 기반으로 경계를 찾고, 피처 값의 단위가 다르면 거리 계산이 왜곡된다.
    print(f"[X_train]", X_train.isnull().sum())
    print(f"[X_test]", X_test.isnull().sum())

    scaler = MinMaxScaler()

    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"[scale_for_svm] X_test_scaled NaN 존재 여부: ", np.sum(np.isnan(X_test_scaled))) # 52건의 NaN 데이터

    return X_train_scaled, X_test_scaled

In [5]:
def train_svm(X_train_scaled, y_train):
    model = SVC(kernel='rbf', random_state=42)

    # 모델 학습
    model.fit(X_train_scaled, y_train)

    # 학습된 모델 반환
    return model

In [10]:
# Random_Forest은 트리들이 독립적으로 학습 -> 결과를 투표로 합산
# 작동 예시: [트리1] [트리2] [트리3] -> 다수결
# 그렇기 때문에 Random_Forest는 병렬처리가 가능하다.
def train_random_forest(X_train, y_train):
    param_grid = {
        "n_estimators": [15, 20, 25, 30],
        "max_depth": [3, 4, 5]
    }

    # 최적화
    grid_search_for_rfc = GridSearchCV(
        RandomForestClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    # 모델 학습
    grid_search_for_rfc.fit(X_train, y_train)

    print(f"[RandomForest] 최적 파라미터: {grid_search_for_rfc.best_params_}")
    print(f"[RandomForest] 교차검증 정확도: {grid_search_for_rfc.best_score_:.4f}")

    # 학습된 모델 반환
    return grid_search_for_rfc.best_estimator_

In [7]:
# Gradient_Boosting은 트리들을 순서대로 학습 -> 앞 트리의 실수를 다음 트리가 보정
# 작동 예시: [트리1] -> 오류 -> [트리2] -> 오류 -> [트리3] -> 최종 합산
# 그렇기 때문에 Gradient_Boosting은 병렬처리가 안된다.
def train_gradient_boosting(X_train, y_train):
    param_grid = {
        "n_estimators": [4, 5, 8],
        "max_depth": [2, 3, 4, 5]
    }

    grid_search_for_gb = GridSearchCV(
        GradientBoostingClassifier(random_state=42),
        param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1
    )

    # 모델 학습
    grid_search_for_gb.fit(X_train, y_train)

    print(f"[GradientBoosting] 최적 파라미터: {grid_search_for_gb.best_params_}")
    print(f"[GradientBoosting] 교차검증 정확도: {grid_search_for_gb.best_score_:.4f}")

    return grid_search_for_gb.best_estimator_

In [8]:
def evaluate_model(model, X_test, y_test, model_name):
    print(f"\n{'='*40}")
    print(f"  {model_name}")
    print(f"{'='*40}")
    # y_test 값은 실제 정답값
    # 예측값 생성
    y_pred = model.predict(X_test)

    # 정확도 출력
    acc = accuracy_score(y_test, y_pred)
    print(f"[{model_name}] Accuracy: {acc:.4f}")

    # 리포트 출력
    print(f"[{model_name}] Classification Report:")
    print(classification_report(y_test, y_pred))

    # 혼동 행렬 출력
    print(f"[{model_name}] Confusion_Matrix")
    print(confusion_matrix(y_test, y_pred))

    return acc

In [9]:
for region in REGION_LIST:
    X_train, X_test, y_train, y_test = load_dataset(region)

    # SVM 모델을 위한 MinMaxScaler
    X_train_scaled, X_test_scaled = scale_for_svm(X_train, X_test)

    # 모델 학습
    svm_model = train_svm(X_train_scaled, y_train)
    rf_model = train_random_forest(X_train, y_train)
    gb_model = train_gradient_boosting(X_train, y_train)

    # 모델 평가
    evaluate_model(svm_model, X_test_scaled, y_test, "SVM MODEL")
    evaluate_model(rf_model, X_test, y_test, "RANDOM FOREST MODEL")
    evaluate_model(gb_model, X_test, y_test, "GRADIENT BOOSTING MODEL")



  [강릉] 데이터셋 로드
학습: 1476행, 테스트: 861행

[train NaN] 없음

[test NaN] 없음
[X_train] Sowing_avg_temp                   0
Sowing_min_temp                   0
Sowing_max_temp                   0
Sowing_d_prcp                     0
Sowing_avg_dew_point_temp         0
Sowing_min_relative_humidity      0
Sowing_avg_relative_humidity      0
Sowing_Total_solar_radiation      0
Sowing_avg_ground_temp            0
Sowing_min_grass_temp             0
Sowing_SPI1                       0
Sowing_SPI2                       0
Sowing_SPI3                       0
Sowing_SPI4                       0
Planting_avg_temp                 0
Planting_min_temp                 0
Planting_max_temp                 0
Planting_d_prcp                   0
Planting_avg_dew_point_temp       0
Planting_min_relative_humidity    0
Planting_avg_relative_humidity    0
Planting_Total_solar_radiation    0
Planting_avg_ground_temp          0
Planting_min_grass_temp           0
Planting_SPI1                     0
Planting_SPI2         